# 04 - Theme mentions over time

**What this does:** reads the cleaned slice saved by **notebook 01**
(`data/processed/posts_slice.parquet`) — so it automatically uses the SAME
time window and subreddits you set there — scans each post's title and body
for a curated list of theme keywords, and counts keyword hits per theme per
day. The chain is: **01 (slice) → 04 (theme counts) → 05 (take-offs)**.

Unlike the ticker notebooks (02/03), this does **not** rely on ticker symbols
being present. A post discussing 'HBM memory pricing' and 'Micron DRAM capacity'
scores hits for the `memory` theme even if it never uses $MU. (That also means
the word-ticker screening from notebook 01 doesn't apply here - themes match
keywords, not tickers.)

Produces two graphs:
1. **Raw keyword hits** - every keyword match counts as 1.
2. **Upvote-weighted hits** - each hit is weighted by the post's upvote score.

Saves `data/processed/daily_theme_counts.parquet`, which notebook 05 uses.

**WARNING: the theme scanner is slow** (~130 posts/sec) - a full year of all
15 subreddits (~2.8M posts) takes hours. Narrow the window in notebook 01 or
set SUBREDDITS there before running this.

Edit `THEME_KEYWORDS` in `src/themes.py` to add or rename themes.

In [ ]:
import os, sys
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
print('project root:', ROOT)

In [ ]:
# ============ PARAMETERS - edit these ============
# NOTE: no TIME WINDOW here any more. The window and subreddits come from
# notebook 01, via the slice file it saves - one source of truth for the chain.
SLICE_PATH        = os.path.join(ROOT, 'data', 'processed', 'posts_slice.parquet')
THEMES_TO_PLOT    = []      # e.g. ['memory', 'ai']; [] = use the TOP_N most mentioned
TOP_N             = 6
DAILY_THEMES_OUT  = os.path.join(ROOT, 'data', 'processed', 'daily_theme_counts.parquet')
# ==================================================

In [ ]:
import pandas as pd
from src.themes import build_daily_theme_counts, THEME_KEYWORDS

# The cleaned slice saved by notebook 01 - THIS is how 04 inherits 01's
# TIME WINDOW and SUBREDDITS. No filters are re-declared here.
if not os.path.exists(SLICE_PATH):
    raise FileNotFoundError(
        'posts_slice.parquet not found - run notebook 01 first '
        '(it saves the cleaned slice this notebook reads).')

posts = pd.read_parquet(SLICE_PATH, columns=['date', 'title', 'selftext', 'score'])
print('posts:', f'{len(posts):,}', '| window:', posts['date'].min(), '->', posts['date'].max())
print('themes defined:', sorted(THEME_KEYWORDS.keys()))

# SLOW: ~130 posts/sec. Narrow the window in notebook 01 if this is too long.
daily = build_daily_theme_counts(posts)
daily.to_parquet(DAILY_THEMES_OUT, index=False)
print('daily theme rows:', len(daily), '-> saved', DAILY_THEMES_OUT)
daily.head()

In [ ]:
import matplotlib.pyplot as plt
daily['date'] = pd.to_datetime(daily['date'])

# Decide which themes to draw.
if THEMES_TO_PLOT:
    chosen = THEMES_TO_PLOT
else:
    totals = daily.groupby('theme')['mention_count'].sum().sort_values(ascending=False)
    chosen = list(totals.head(TOP_N).index)
print('plotting:', chosen)

def plot_signal(column, title, ylabel):
    plt.figure(figsize=(13, 6))
    for theme in chosen:
        one = daily[daily['theme'] == theme].sort_values('date')
        plt.plot(one['date'], one[column], marker='o', markersize=3, label=theme)
    plt.title(title)
    plt.xlabel('date'); plt.ylabel(ylabel)
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.show()

In [ ]:
# GRAPH 1 - raw keyword hits per theme per day
plot_signal('mention_count', 'Raw theme keyword hits over time', 'keyword hits per day')

In [ ]:
# GRAPH 2 - upvote-weighted hits (each hit weighted by the post's upvotes)
plot_signal('weighted_count', 'Upvote-weighted theme mentions over time', 'sum of upvotes per day')

In [ ]:
# Summary table: total keyword hits per theme across all time
summary = (
    daily.groupby('theme')
    .agg(total_hits=('mention_count', 'sum'),
         weighted_hits=('weighted_count', 'sum'),
         days_active=('date', 'nunique'))
    .sort_values('total_hits', ascending=False)
)
summary